# LLM-as-Judge — Semantic Quality Evaluation

Evaluates context injection benchmark results using an LLM as a semantic judge.

**Why**: Structural validation catches formatting errors but misses whether the query actually matches the user's intent. This notebook uses a judge LLM to give a binary pass/fail on each semantic dimension.

**Scoring**: Each dimension is **binary (0 or 1)** — the query either satisfies the criterion or it doesn't. Total score: 0–5.

**Judge backends**: Claude API or local LLM via Ollama.

**Input**: `context_benchmark_results.csv`  
**Output**: `judge_results.csv`

In [ ]:
import sys
import os
import json
import time
import pandas as pd
from datetime import timedelta

sys.path.insert(0, os.path.join(os.getcwd(), '..'))

# ── Configuration ─────────────────────────────────────
BENCHMARK_CSV = "context_benchmark_results.csv"
OUTPUT_CSV    = "judge_results.csv"
OVERWRITE     = True
BATCH_DELAY   = 0.5        # seconds between API calls

# ── Judge backend: "claude" or "ollama" ────────────────
JUDGE_BACKEND = "ollama"    # Change to "ollama" for local

# Claude settings (used when JUDGE_BACKEND == "claude")
CLAUDE_MODEL  = "claude-sonnet-4-5-20250929"

# Ollama settings (used when JUDGE_BACKEND == "ollama")
OLLAMA_MODEL  = "mistral-small3.1"           # or gemma2:27b, mistral-small3.1, etc.
OLLAMA_URL    = "http://localhost:11434"

print(f"Judge backend: {JUDGE_BACKEND}")
print(f"  Model: {CLAUDE_MODEL if JUDGE_BACKEND == 'claude' else OLLAMA_MODEL}")

In [ ]:
df = pd.read_csv(BENCHMARK_CSV)
print(f"Loaded {len(df)} rows: {df['Model'].nunique()} models × {df['Input'].nunique()} queries × {df['Mode'].nunique()} modes")
df.head(3)

## Judge Prompt — Binary Scoring (0/1)

| Dimension | 0 (Fail) | 1 (Pass) |
|---|---|---|
| **Term accuracy** | Key terms missing, wrong, or hallucinated | All relevant terms correctly represented |
| **Category correctness** | Wrong domain or missing when clearly needed | Correct categories (or correctly NONE) |
| **Author handling** | Hallucinated authors or missed named ones | Authors correct (or correctly NONE) |
| **Date/scope** | Wrong dates or phantom dates added | Dates correct or correctly omitted |
| **Intent match** | Query misses the user's intent | Query captures what the user asked for |

In [ ]:
JUDGE_SYSTEM_PROMPT = """You are an expert evaluator of arXiv API search queries.

You will be given a user's natural language request and the arXiv query that was generated.

## arXiv Query Syntax Reference
- `all:"term"` — search all fields
- `ti:"term"` — title search
- `au:Lastname` or `au:Lastname_X` — author search
- `cat:code` — category filter (e.g., cat:astro-ph.GA, cat:cs.LG)
- `submittedDate:[YYYYMMDD TO YYYYMMDD]` — date range
- `AND`, `OR`, `ANDNOT` — boolean operators
- `NONE` — component intentionally empty

## Scoring — Binary (0 or 1) on 5 dimensions

### 1. term_accuracy (content component)
- 0: Key search terms are missing, wrong, or hallucinated
- 1: All relevant terms from the user request are correctly represented

### 2. category_correctness (category component)
- 0: Wrong domain (e.g., cs.LG for astrophysics), or NONE when a specific category was clearly needed
- 1: Correct arXiv categories for the topic, or correctly NONE when the query is broad

### 3. author_handling (author component)
- 0: Hallucinated authors, or missed explicitly named authors/team members
- 1: Authors correctly handled, or correctly NONE when no authors were requested
- Note: When external context defines a team, the author component should list team members

### 4. date_scope
- 0: Dates wrong, phantom dates added when user didn't mention dates, or dates missing when user specified them
- 1: Correct date range when specified, or correctly omitted when user didn't mention dates
- Note: When context defines a project phase, dates should match that phase

### 5. intent_match (overall)
- 0: Query would mostly retrieve irrelevant papers
- 1: Query is reasonably targeted to what the user asked for

## Response Format

Respond with ONLY a JSON object. No markdown backticks, no preamble:

{"term_accuracy": 0, "category_correctness": 0, "author_handling": 0, "date_scope": 0, "intent_match": 0, "total": 0, "flags": [], "reasoning": "Brief explanation"}

Possible flags: "hallucinated_authors", "wrong_domain", "missed_date", "phantom_date", "team_not_expanded", "acronym_not_resolved", "over_expanded"
"""

print(f"Judge system prompt: {len(JUDGE_SYSTEM_PROMPT)} chars")

In [ ]:
# ── Context reference for the judge ──────────────────────────────
CONTEXT_REFERENCE = {
    "glossary": {
        "LINER": "Low-Ionization Nuclear Emission-Line Region (NOT AGN). Category: astro-ph.GA",
        "BPT": "Baldwin-Phillips-Terlevich diagram. Category: astro-ph.GA",
        "XRB": "X-ray Binary. Category: astro-ph.HE",
        "ICM": "Intracluster Medium. Category: astro-ph.CO, astro-ph.GA",
        "SED": "Spectral Energy Distribution. Category: astro-ph.GA, astro-ph.SR",
        "BAO": "Baryon Acoustic Oscillations. Category: astro-ph.CO",
        "RLHF": "Reinforcement Learning from Human Feedback. Category: cs.LG, cs.CL",
        "RAG": "Retrieval-Augmented Generation. Category: cs.CL, cs.IR",
        "CoT": "Chain of Thought. Category: cs.CL, cs.AI",
        "tSZ": "thermal Sunyaev-Zel'dovich effect. Category: astro-ph.CO",
    },
    "teams": {
        "NTE": "María López Sánchez, Roberto Fulánez Santos, Ignacio Ordovás Pascual, Paul Martin-Dussaud",
        "DL4Astro": "Ignacio Ordovás Pascual, Wei Zhang, Ana García Pérez",
        "QML": "Paul Martin-Dussaud, Elena Rodríguez, Kenji Tanaka",
    },
    "project_phases": {
        "NX Project Phase 1": "2021-01-01 to 2022-06-30",
        "NX Project Phase 2": "2022-07-01 to 2024-12-31",
    },
    "instruments": {
        "eROSITA": "X-ray telescope. Categories: astro-ph.HE, astro-ph.CO",
    },
    "datasets": {
        "SDSS": "Sloan Digital Sky Survey. Categories: astro-ph.GA, astro-ph.CO",
        "DESI": "Dark Energy Spectroscopic Instrument. Categories: astro-ph.CO",
    },
}


def build_judge_user_prompt(row):
    """Build the user prompt for the judge from a benchmark result row."""
    mode = row['Mode']
    ctx_types = str(row.get('ContextTypes', ''))

    context_section = ""
    if mode == 'context' and ctx_types and ctx_types not in ('nan', 'control'):
        context_section = "\n## External Context Provided to the LLM\n"
        for ctx_type in ctx_types.split(', '):
            ctx_type = ctx_type.strip()
            ref = CONTEXT_REFERENCE.get(ctx_type, CONTEXT_REFERENCE.get(ctx_type + 's', {}))
            if isinstance(ref, dict):
                for term, defn in ref.items():
                    if term.lower() in row['Input'].lower():
                        context_section += f"- {term}: {defn}\n"
            # Project phases — match loosely
            if ctx_type == 'project_phase':
                for phase, dates in CONTEXT_REFERENCE['project_phases'].items():
                    if any(w in row['Input'].lower() for w in ['phase', 'nx project']):
                        context_section += f"- {phase}: {dates}\n"
            if ctx_type in ('instrument', 'dataset'):
                section_key = ctx_type + 's'
                for key, desc in CONTEXT_REFERENCE.get(section_key, {}).items():
                    if key.lower() in row['Input'].lower():
                        context_section += f"- {key}: {desc}\n"
            if ctx_type == 'team':
                for team, members in CONTEXT_REFERENCE['teams'].items():
                    if team.lower() in row['Input'].lower():
                        context_section += f"- Team {team}: {members}\n"

    content  = str(row.get('Content', 'NONE'))
    author   = str(row.get('Author', 'NONE'))
    category = str(row.get('Category', 'NONE'))
    combined = str(row.get('Query', ''))
    papers   = row.get('Papers', 0)

    return f"""## User Request
{row['Input']}

## Mode
{mode} {'(no external context)' if mode == 'baseline' else '(context injected)'}
{context_section}
## Generated Query Components
- Content: {content}
- Author: {author}
- Category: {category}
- Combined: {combined}
- Papers retrieved: {papers}

Score this query on the 5 binary dimensions (0 or 1 each). Respond with ONLY a JSON object."""


# Quick test
print(build_judge_user_prompt(df.iloc[0]))

## Judge Backends

In [ ]:
def _parse_judge_response(raw_text):
    """Parse JSON from judge response, handling markdown fences."""
    text = raw_text.strip()
    text = text.replace('```json', '').replace('```', '').strip()
    # Some local models wrap in <think>...</think> tags
    if '<think>' in text:
        # Extract everything after the last </think>
        parts = text.split('</think>')
        text = parts[-1].strip()
    result = json.loads(text)

    expected = ['term_accuracy', 'category_correctness', 'author_handling', 'date_scope', 'intent_match']
    for key in expected:
        if key not in result:
            result[key] = -1
        else:
            # Clamp to 0/1 binary
            val = result[key]
            if isinstance(val, (int, float)):
                result[key] = 1 if val >= 1 else 0

    scores = [result.get(k, 0) for k in expected]
    result['total'] = sum(s for s in scores if s >= 0)
    result['parse_error'] = False
    return result


def _make_error_result(error_msg):
    return {
        'term_accuracy': -1, 'category_correctness': -1,
        'author_handling': -1, 'date_scope': -1, 'intent_match': -1,
        'total': -1, 'flags': [], 'reasoning': error_msg, 'parse_error': True,
    }


# ── Claude backend ────────────────────────────────────
def _judge_claude(user_prompt, retries=2):
    import anthropic
    client = anthropic.Anthropic()

    for attempt in range(retries + 1):
        try:
            response = client.messages.create(
                model=CLAUDE_MODEL, max_tokens=500,
                system=JUDGE_SYSTEM_PROMPT,
                messages=[{"role": "user", "content": user_prompt}],
            )
            return _parse_judge_response(response.content[0].text)
        except json.JSONDecodeError:
            if attempt < retries: time.sleep(1); continue
            return _make_error_result(f"JSON parse error after {retries+1} attempts")
        except Exception as e:
            if attempt < retries: time.sleep(2); continue
            return _make_error_result(f"Claude API error: {e}")


# ── Ollama backend ────────────────────────────────────
def _judge_ollama(user_prompt, retries=2):
    import ollama
    client = ollama.Client(host=OLLAMA_URL)

    full_prompt = f"{JUDGE_SYSTEM_PROMPT}\n\n{user_prompt}"

    for attempt in range(retries + 1):
        try:
            response = client.generate(
                model=OLLAMA_MODEL,
                prompt=full_prompt,
                stream=False,
                options={"temperature": 0.1, "num_predict": 512},
            )
            raw_text = response.get('response', '')
            return _parse_judge_response(raw_text)
        except json.JSONDecodeError:
            if attempt < retries: time.sleep(1); continue
            return _make_error_result(f"JSON parse error after {retries+1} attempts")
        except Exception as e:
            if attempt < retries: time.sleep(2); continue
            return _make_error_result(f"Ollama error: {e}")


def judge_single_row(row):
    """Route to the configured judge backend."""
    user_prompt = build_judge_user_prompt(row)
    if JUDGE_BACKEND == "claude":
        return _judge_claude(user_prompt)
    elif JUDGE_BACKEND == "ollama":
        return _judge_ollama(user_prompt)
    else:
        raise ValueError(f"Unknown JUDGE_BACKEND: {JUDGE_BACKEND}")


# Quick test
test_result = judge_single_row(df.iloc[0])
print(json.dumps(test_result, indent=2))

## Run Judge

In [ ]:
existing_judged = set()
if not OVERWRITE and os.path.exists(OUTPUT_CSV):
    existing = pd.read_csv(OUTPUT_CSV)
    existing_judged = set(zip(existing['Model'], existing['Input'], existing['Mode']))
    print(f"Loaded {len(existing_judged)} existing judgments — will skip these.")

results = []
n_total = len(df)
t_start = time.time()
parse_errors = 0

for i, (idx, row) in enumerate(df.iterrows()):
    key = (row['Model'], row['Input'], row['Mode'])
    if key in existing_judged:
        continue

    judgment = judge_single_row(row)

    result = {
        'Model': row['Model'], 'Size': row.get('Size', ''),
        'Mode': row['Mode'], 'Input': row['Input'],
        'ContextTypes': row.get('ContextTypes', ''),
        'Content': row.get('Content', ''), 'Author': row.get('Author', ''),
        'Category': row.get('Category', ''),
        'Papers': row.get('Papers', 0), 'Valid': row.get('Valid', False),
        'J_term': judgment['term_accuracy'],
        'J_category': judgment['category_correctness'],
        'J_author': judgment['author_handling'],
        'J_date': judgment['date_scope'],
        'J_intent': judgment['intent_match'],
        'J_total': judgment['total'],
        'J_flags': '|'.join(judgment.get('flags', [])),
        'J_reasoning': judgment.get('reasoning', ''),
        'J_parse_error': judgment.get('parse_error', False),
        'J_backend': JUDGE_BACKEND,
        'J_model': CLAUDE_MODEL if JUDGE_BACKEND == 'claude' else OLLAMA_MODEL,
    }
    results.append(result)

    if judgment.get('parse_error'): parse_errors += 1

    done = i + 1
    elapsed = time.time() - t_start
    eta = (elapsed / done) * (n_total - done)

    if done % 10 == 0 or done == 1:
        print(f"  [{done:>3}/{n_total}] {row['Model']:<20s} {row['Mode']:<10s} "
              f"score={judgment['total']}/5  ETA {timedelta(seconds=int(eta))}")

    time.sleep(BATCH_DELAY)

print(f"\nDone: {len(results)} judgments in {timedelta(seconds=int(time.time() - t_start))}")
print(f"Parse errors: {parse_errors}/{len(results)}")

In [ ]:
judge_df = pd.DataFrame(results)

if not OVERWRITE and os.path.exists(OUTPUT_CSV):
    existing = pd.read_csv(OUTPUT_CSV)
    judge_df = pd.concat([existing, judge_df], ignore_index=True)

judge_df.to_csv(OUTPUT_CSV, index=False)
print(f"Saved {len(judge_df)} rows to {OUTPUT_CSV}")
judge_df.head()

## Analysis: Overall Scores

In [ ]:
judge_df = pd.read_csv(OUTPUT_CSV)
valid_judge = judge_df[~judge_df['J_parse_error']].copy()
print(f"Valid judgments: {len(valid_judge)}/{len(judge_df)}")

print("\n" + "=" * 60)
print("  OVERALL — Baseline vs Context (binary scoring, /5)")
print("=" * 60)

dims = [
    ('J_term', 'Term accuracy'),
    ('J_category', 'Category'),
    ('J_author', 'Author handling'),
    ('J_date', 'Date/scope'),
    ('J_intent', 'Intent match'),
    ('J_total', 'TOTAL (/5)'),
]

for mode in ['baseline', 'context']:
    sub = valid_judge[valid_judge['Mode'] == mode]
    print(f"\n  {mode.upper()} ({len(sub)} rows)")
    for col, label in dims:
        mean = sub[col].mean()
        mx = 1 if col != 'J_total' else 5
        pct = mean / mx * 100
        print(f"    {label:<20s} {mean:.2f}/{mx}  ({pct:.0f}%)")

## Analysis: Per-Model Comparison

In [ ]:
score_cols = ['J_term', 'J_category', 'J_author', 'J_date', 'J_intent', 'J_total']

bl = valid_judge[valid_judge['Mode'] == 'baseline'].groupby('Model')[score_cols].mean()
ct = valid_judge[valid_judge['Mode'] == 'context'].groupby('Model')[score_cols].mean()

comp = pd.DataFrame({
    'BL /5': bl['J_total'].round(2),
    'CTX /5': ct['J_total'].round(2),
    'Delta': (ct['J_total'] - bl['J_total']).round(2),
    'BL_term': bl['J_term'].round(2),
    'CTX_term': ct['J_term'].round(2),
    'BL_auth': bl['J_author'].round(2),
    'CTX_auth': ct['J_author'].round(2),
}).sort_values('CTX /5', ascending=False)

print("Per-model judge scores (mean):")
print(comp.to_string())

## Analysis: Per-Query Comparison

In [ ]:
print("\n" + "=" * 90)
print("  PER-QUERY — Baseline vs Context")
print("=" * 90)

for input_text in valid_judge['Input'].unique():
    q = valid_judge[valid_judge['Input'] == input_text]
    bl_q = q[q['Mode'] == 'baseline']
    ct_q = q[q['Mode'] == 'context']

    bl_mean = bl_q['J_total'].mean()
    ct_mean = ct_q['J_total'].mean()
    delta = ct_mean - bl_mean

    ctx_types = q['ContextTypes'].iloc[0]
    indicator = "↑" if delta > 0.3 else "↓" if delta < -0.3 else "→"

    print(f"  {indicator} {input_text[:55]:<55s}  BL={bl_mean:3.1f}  CTX={ct_mean:3.1f}  Δ={delta:+3.1f}  [{ctx_types}]")

## Analysis: Judge vs Structural Validator

In [ ]:
n_valid = len(valid_judge[valid_judge['Valid'] == True])
n_invalid = len(valid_judge[valid_judge['Valid'] == False])

# Structurally valid but semantically poor (judge <= 2/5)
valid_but_bad = valid_judge[(valid_judge['Valid'] == True) & (valid_judge['J_total'] <= 2)]
print(f"Structurally VALID but judge ≤ 2/5: {len(valid_but_bad)}/{n_valid}")
if not valid_but_bad.empty:
    for _, r in valid_but_bad.head(5).iterrows():
        print(f"  [{r['Model']:<18s} {r['Mode']:<8s}] {r['J_total']}/5  {r['Input'][:50]}")

# Structurally invalid but semantically good (judge >= 4/5)
invalid_but_good = valid_judge[(valid_judge['Valid'] == False) & (valid_judge['J_total'] >= 4)]
print(f"\nStructurally INVALID but judge ≥ 4/5: {len(invalid_but_good)}/{n_invalid}")
if not invalid_but_good.empty:
    for _, r in invalid_but_good.head(5).iterrows():
        print(f"  [{r['Model']:<18s} {r['Mode']:<8s}] {r['J_total']}/5  {r['Input'][:50]}")

## Analysis: Flag Frequency

In [ ]:
for mode in ['baseline', 'context']:
    sub = valid_judge[valid_judge['Mode'] == mode]
    all_flags = []
    for flags_str in sub['J_flags'].fillna(''):
        if flags_str:
            all_flags.extend(flags_str.split('|'))

    if not all_flags:
        print(f"\n{mode.upper()}: No flags")
        continue

    flag_counts = pd.Series(all_flags).value_counts()
    print(f"\n{mode.upper()} ({len(sub)} rows):")
    for flag, count in flag_counts.items():
        print(f"  {flag:<30s} {count:>3} ({count/len(sub)*100:.0f}%)")

## Summary Table (for the article)

In [ ]:
summary_rows = []
for size in ['tiny', 'small', 'medium', 'large', 'cloud']:
    sub = valid_judge[valid_judge['Size'] == size]
    if sub.empty: continue
    bl_sub = sub[sub['Mode'] == 'baseline']
    ct_sub = sub[sub['Mode'] == 'context']

    summary_rows.append({
        'Size': size.upper(),
        'Models': sub['Model'].nunique(),
        'BL structural': f"{bl_sub['Valid'].mean()*100:.0f}%",
        'CTX structural': f"{ct_sub['Valid'].mean()*100:.0f}%",
        'BL judge /5': f"{bl_sub['J_total'].mean():.1f}",
        'CTX judge /5': f"{ct_sub['J_total'].mean():.1f}",
        'Delta': f"{ct_sub['J_total'].mean() - bl_sub['J_total'].mean():+.1f}",
    })

print(pd.DataFrame(summary_rows).to_string(index=False))
print(f"\nJudge backend: {JUDGE_BACKEND} ({CLAUDE_MODEL if JUDGE_BACKEND == 'claude' else OLLAMA_MODEL})")

## Heatmap: Judge Scores by LLM Size

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Dimension labels (shortened for display)
dimension_labels = {
    'J_term': 'Term accuracy',
    'J_category': 'Category',
    'J_author': 'Author handling',
    'J_date': 'Date/scope',
    'J_intent': 'Intent match',
}

dimension_cols = ['J_term', 'J_category', 'J_author', 'J_date', 'J_intent']

# Create heatmaps for both baseline and context modes
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax_idx, mode in enumerate(['baseline', 'context']):
    mode_data = valid_judge[valid_judge['Mode'] == mode]
    
    # Pivot: rows = dimensions, columns = sizes
    size_order = ['tiny', 'small', 'medium', 'large', 'cloud']
    
    # Build matrix manually
    sizes_present = sorted(mode_data['Size'].unique())
    matrix = []
    
    for dim_col in dimension_cols:
        row = []
        for size in size_order:
            sub = mode_data[mode_data['Size'] == size]
            if len(sub) > 0:
                mean_val = sub[dim_col].mean()
                row.append(mean_val)
            else:
                row.append(np.nan)
        matrix.append(row)
    
    matrix = np.array(matrix)
    
    # Plot heatmap
    ax = axes[ax_idx]
    sns.heatmap(
        matrix,
        annot=True,  # Show values
        fmt='.2f',   # 2 decimal places
        cmap='RdYlGn',  # Red-Yellow-Green
        vmin=0,
        vmax=1,
        cbar_kws={'label': 'Mean Score'},
        ax=ax,
        xticklabels=size_order,
        yticklabels=[dimension_labels[col] for col in dimension_cols],
        linewidths=0.5,
        linecolor='gray',
    )
    
    ax.set_title(f'{mode.replace("baseline", "BASELINE").replace("context", "CONTEXT")} Mode', fontsize=12, fontweight='bold')
    ax.set_xlabel('LLM Size', fontsize=10)
    ax.set_ylabel('Judge Dimension', fontsize=10)

plt.tight_layout()
plt.savefig('judge_heatmap_by_size.png', dpi=150, bbox_inches='tight')
plt.show()

print("Heatmap saved to judge_heatmap_by_size.png")